# Benchmarking the Tuna devices performance using common measurement routines

**Author:** Marios Samiotis (m.samiotis@tudelft.nl)

**Date:** April 29, 2026

**Last updated:** April 29, 2026

# 1. Introduction

In this notebook we will use some of the most common benchmark measurement routines to assess the performance of any of the Tuna superconducting quantum backends which exist at the [Quantum Inspire 2.0](https://www.quantum-inspire.com/) cloud services.

Prior to using this notebook, for optimal performance, make sure that you are running it within a customized Python 3.12 environment which includes the packages "quantuminspire" and "qiskit-quantuminspire".

For details on how to create such a Python environment, follow the installation instructions in the [README file](https://github.com/DiCarloLab-Delft/QuantumInspireUtilities/blob/main/README.md).

Useful links:
1. [Tuna Backends Operational Specifics](https://www.quantum-inspire.com/kbase/tuna-operational-specifics/)
2. [Backend Performance Dashboards](https://monitoring.qutech.support/public-dashboards/c494a21fb6b7405f850ab8f340f798ef?orgId=1&refresh=10s) [live updates]
3. [Join our Slack community!](https://join.slack.com/t/qisuperconducting/shared_invite/zt-35o7zitdh-_9QPmB53hhLy12Eat5gwWA)

First, we run the following cell in order to login to the Quantum Inspire platform. You need an account in order to login to the platform.

Please click on "Confirm" in the pop-up browser window to login.

In [ ]:
! qi login "https://api.quantum-inspire.com"

Run the following cell for all necessary library imports,

In [ ]:
import numpy as np
from qiskit_quantuminspire.qi_provider import QIProvider
from qi_utilities.device_simulation.simulators import NoisySimulator # use the simulator in case all backends are unavailable!
from qi_utilities.measurements.device_control import DeviceControl

# 2. Connecting to the backend

We first connect to the Quantum Inspire provider, by running the following cell,

In [ ]:
provider = QIProvider()

The following command prints the full list of all available backends of the platform,

In [ ]:
provider.backends()

We now connect to a Tuna backend by running the following cell,

In [ ]:
backend_name = "Tuna-9"
backend = provider.get_backend(name=backend_name)

In [ ]:
backend = NoisySimulator("Starmon-7")

By running the cell below, the connectivity of the processor is shown (otherwise known as "coupling map"),

In [ ]:
backend.coupling_map.draw()

Any single- or multi-qubit gates that may be applied on the qubits of the Tuna backends are referred to as "instructions". The full list of allowed instructions of the Tuna backend is shown by running the following cell,

In [ ]:
backend.instructions

When executing a quantum algorithm on any backend, one needs to choose the number of measurement shots to be performed for the given quantum circuit. The maximum number of allowed shots on the Tuna backend can be obtained by running the command

In [ ]:
backend.max_shots

# DeviceControl class

In [ ]:
device = DeviceControl(backend)

In [ ]:
device.measure_rabi(qubit_list='all', # [0, 5, 6]
                    rotation_angles = np.linspace(-np.pi, np.pi, num=51),
                    rotation_axis='x',
                    num_shots = 2**12,
                    use_ro_cal_points=True)

In [ ]:
device.measure_T1(qubit_list='all', # [0, 5, 6]
                  measurement_times=np.linspace(0, 50e-6, num=41),
                  num_shots=2**12)

In [ ]:
device.measure_T2_ramsey(qubit_list='all', # [0, 8]
                         measurement_times=np.linspace(0, 1e-6, num=61),
                         num_shots=2**12)

In [ ]:
device.measure_T2_echo(qubit_list='all', # [0, 8]
                       measurement_times=np.linspace(0, 1e-6, num=61),
                       num_shots=2**12)

In [ ]:
device.measure_flipping(qubit_list='all',
                        num_shots=2**12,
                        max_number_of_flips=30,
                        equator=True)

In [ ]:
device.measure_allxy(qubit_list='all', # [0, 8]
                     num_shots=2**12)

In [ ]:
device.measure_conditional_oscillation(qubit_pairs = [[0, 2]],
                                       num_angles=19,
                                       num_shots = 2**12,
                                       cz_repetitions=1)

In [ ]:
device.measure_bell_state_fidelity(qubit_pairs = [[0, 2]],
                                   bell_state = "phi_plus",
                                   num_shots = 2**14)

In [ ]:
device.num_shots

In [ ]:
device.latest_qc.draw('mpl')

# Development

In [ ]:
from qi_utilities.measurements.measurement_classes import *

In [ ]:
import matplotlib.pyplot as plt

rabi_loaded_data = RabiMeasurement.load_data(job_timestamp='20261106_152350')
experiment_name = rabi_loaded_data['Experiment name']
experiment_timestamp = rabi_loaded_data['Experiment timestamp']
backend_name = rabi_loaded_data['Backend name']
num_shots = rabi_loaded_data['Number of shots']
qubit = 'Q0'

rotation_angles = rabi_loaded_data['Rotation angles [rad]']
prob_excited = [rabi_loaded_data['Processed data'][qubit][meas_idx]['1'] for meas_idx in range(len(rabi_loaded_data['Processed data'][qubit]))]

fig, ax = plt.subplots(figsize=(18, 5), dpi=300)
ax.plot(rotation_angles,
            prob_excited,
            label=f'Qubit {qubit}',
            alpha=0.6,
            color = 'C0',
            marker='o')
ax.set_xlabel('Applied rotation')
ax.set_ylabel(r'Population $|1\rangle$')
ax.set_title(
    f'{experiment_name}'
    f'\n{backend_name} processor | num_shots = {num_shots}'
    f'\n{experiment_timestamp}')
plt.show()

In [ ]:
T1_Measurement.load_data(job_timestamp='20260703_172723')

In [ ]:
FlippingMeasurement.load_data(job_timestamp='20260703_173524')

In [ ]:
AllXYMeasurement.load_data(job_timestamp='20260703_173638')

In [ ]:
ConditionalOscMeasurement.load_data(job_timestamp='20260703_173733')